### **Data Cleaning Dataset Job_posting**
Nama : Mohammad Wildan Abdurrahman

E-mail : wildanabd74@gmail.com

ID Dicoding : CDCC358D6Y0569

##**Import semua library yang digunakan**

In [1]:

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

# **Data Wrangling**

Membaca dataset yang diunggah di google collabs

In [3]:
df = pd.read_csv ('/content/job_postings.csv')
df.head()
print(df.columns.tolist())

['job_id', 'company_id', 'title', 'description', 'max_salary', 'med_salary', 'min_salary', 'pay_period', 'formatted_work_type', 'location', 'applies', 'original_listed_time', 'remote_allowed', 'views', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'scraped']


menghapus kolom yang tidak diperlukan

In [4]:
kolom_yang_dihapus = [
    'max_salary', 'med_salary', 'min_salary', 'pay_period', 'applies',
    'original_listed_time', 'remote_allowed', 'views',
    'application_type', 'expiry', 'closed_time', 'listed_time',
    'posting_domain', 'sponsored', 'currency',
    'compensation_type', 'scraped'
]
df = df.drop(columns=kolom_yang_dihapus, errors='ignore')

print("Kolom yang sekarang ada di dataset:")
print(df.columns.tolist())

df.head()

Kolom yang sekarang ada di dataset:
['job_id', 'company_id', 'title', 'description', 'formatted_work_type', 'location', 'job_posting_url', 'application_url', 'formatted_experience_level', 'skills_desc', 'work_type']


,job_id,company_id,title,description,formatted_work_type,location,job_posting_url,application_url,formatted_experience_level,skills_desc,work_type
0,2371637339,3199778.0,Sales Executive,"Are you a smart, authentic, pro-active B2B Sal...",Full-time,"Central Jakarta, Jakarta, Indonesia",https://www.linkedin.com/jobs/view/2371637339/...,NaN,NaN,NaN,FULL_TIME
1,3486411410,14512264.0,Partnership Associate,Responsibilities :Building strong strategic re...,Full-time,"South Jakarta, Jakarta, Indonesia",https://www.linkedin.com/jobs/view/3486411410/...,NaN,NaN,NaN,FULL_TIME
2,3506214686,NaN,Human Resources Coordinator,🚛 We’re Hiring: Fleet Supervisor!\nPT Surya Mi...,Full-time,"Surabaya, East Java, Indonesia",https://www.linkedin.com/jobs/view/3506214686/...,NaN,NaN,NaN,FULL_TIME
3,3611210067,6451760.0,Finance Accounting Specialist,Job Description\n\nEngage in the monthly finan...,Full-time,"Jakarta, Indonesia",https://www.linkedin.com/jobs/view/3611210067/...,https://careers.shopee.sg/job-detail/J00160745...,Mid-Senior level,NaN,FULL_TIME
4,3645884531,109026465.0,Sales Marketing for Marine & Offshore,Kualifikasi:Pendidikan minimal D3/S1 di bidang...,Full-time,Jakarta Metropolitan Area,https://www.linkedin.com/jobs/view/3645884531/...,NaN,NaN,NaN,FULL_TIME


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4085 entries, 0 to 4084
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   job_id                      4085 non-null   int64  
 1   company_id                  4028 non-null   float64
 2   title                       4085 non-null   object 
 3   description                 4075 non-null   object 
 4   formatted_work_type         4085 non-null   object 
 5   location                    4085 non-null   object 
 6   job_posting_url             4085 non-null   object 
 7   application_url             1833 non-null   object 
 8   formatted_experience_level  1138 non-null   object 
 9   skills_desc                 74 non-null     object 
 10  work_type                   4085 non-null   object 
dtypes: float64(1), int64(1), object(9)
memory usage: 351.2+ KB


**Insight**

Dataset job_postings.csv telah berhasil dimuat dengan total 4.085 entri dan telah diseleksi menjadi 9 kolom utama.

Kondisi Data Kosong: Beberapa kolom memiliki nilai null yang signifikan, seperti skills_desc (4.011 null), formatted_experience_level (2.947 null), dan description (10 null), yang perlu ditangani sebelum analisis lebih lanjut.

Dataset mencakup informasi penting seperti identitas pekerjaan, judul, lokasi, serta deskripsi lengkap, yang memungkinkan sistem memberikan rekomendasi link pekerjaan yang sesuai dengan kualifikasi pada CV pengguna.

**Assesing Data**

Pada tahap ini, kita melakukan pemeriksaan awal terhadap dataset untuk mengidentifikasi potensi masalah yang dapat memengaruhi analisis. Assessing data bertujuan untuk menemukan inkonsistensi, missing values, dancduplikasidalam data.

cek inkonsistensi pada kolom title



In [6]:

# Regex ini mencari simbol $ atau Rp yang diikuti oleh angka
# \d+ artinya harus ada angka setelah simbol tersebut
currency_pattern = r'(\$\s?\d+|Rp\s?\d+|IDR\s?\d+)'

# Kita gunakan str.contains dengan pola yang lebih ketat
title_with_salary = df[df['title'].str.contains(currency_pattern, regex=True, na=False)]

print(f"--- [7] Cek Anomali: Judul yang FIX mengandung Gaji ---")
print(f"Total ditemukan: {len(title_with_salary)} data\n")

if not title_with_salary.empty:
    # Menampilkan judul agar kita bisa lihat polanya
    display(title_with_salary[['title', 'location']])
else:
    print("Tidak ditemukan judul dengan nominal gaji.")

--- [7] Cek Anomali: Judul yang FIX mengandung Gaji ---
Total ditemukan: 20 data



/tmp/ipykernel_11510/2601849443.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  title_with_salary = df[df['title'].str.contains(currency_pattern, regex=True, na=False)]


,title,location
490,Integration Software Engineer | $80 Remote,APJ
521,Fire Service Supervisor | $75/hr | Remote,APJ
673,Senior AI Product Platform Engineer @ High Gro...,"Denpasar, Bali, Indonesia"
745,LibreSprite Pixel Artist – Game Art & Animatio...,APJ
748,Pixel Artist -LibreSprite (Animation & Sprites...,APJ
790,Pixel Art Animator | $65/hr Remote,APJ
1206,HR Policy & Compliance Specialist | $36/hr Remote,APJ
1259,Audio Engineer (Audio QA & Restoration) | $55/...,APJ
1361,HR Policy & Compliance Specialist | $36/hr Remote,Indonesia
1485,Motion Data Collection Specialist- Video & Sen...,APJ


Judul yang ada mata uangnya, nanti perlu dihilangkan mata uang dan gajinya

cek inkonsistensi pada kolom formated_work_type

In [7]:
# Menampilkan semua jenis tipe kerja yang ada
print(df['formatted_work_type'].unique())
print("\nJumlah per kategori:")
print(df['formatted_work_type'].value_counts())

['Full-time' 'Internship' 'Contract' 'Temporary' 'Volunteer' 'Part-time'
 'Other']

Jumlah per kategori:
formatted_work_type
Full-time     3157
Contract       652
Internship     134
Part-time       96
Temporary       20
Other           20
Volunteer        6
Name: count, dtype: int64


Data di kolom ini sudah sangat rapi. Tidak ada kesalahan tulis

Selannjutnya cek inkonsistensi pada kolom location

In [8]:
print(f"Total Lokasi Unik: {df['location'].nunique()}")
# Menampilkan 30 lokasi terpopuler
print(df['location'].value_counts().head(30))

Total Lokasi Unik: 308
location
Jakarta, Indonesia                     867
Jakarta Metropolitan Area              458
Jakarta, Jakarta, Indonesia            355
Indonesia                              234
APAC                                   207
Tangerang, Banten, Indonesia           136
South Jakarta, Jakarta, Indonesia       81
Surabaya, East Java, Indonesia          79
Bandung, West Java, Indonesia           77
Batam, Riau Islands, Indonesia          76
Denpasar, Bali, Indonesia               72
Bali, Indonesia                         63
APJ                                     61
Gambir, Jakarta, Indonesia              56
Semarang, Central Java, Indonesia       43
Bogor, West Java, Indonesia             40
Lembang, West Java, Indonesia           40
Yogyakarta, Yogyakarta, Indonesia       30
Southeast Asia                          25
West Jakarta, Jakarta, Indonesia        24
Palembang, South Sumatra, Indonesia     23
Greater Semarang                        23
West Karawang, West Ja

keterangan Indonesia sebaiknya dihapus, karena hampir 90% data berada di Indonesia
untuk wilayah (South, North, dsb) diubah ke Bahasa Indonesia (Selatan, Utara, dsb, dan Jakarta Metropolitan Area diubah menjadi istilah yang lebih familiar, yaitu Jabodetabek.

Selanjutnya, cek konsistensi untuk kategori pengalaman

In [9]:
# Menampilkan kategori level pengalaman
print(df['formatted_experience_level'].unique())
print("\nJumlah per kategori (termasuk yang kosong):")
print(df['formatted_experience_level'].value_counts(dropna=False))

[nan 'Mid-Senior level' 'Entry level' 'Associate' 'Internship' 'Director'
 'Executive']

Jumlah per kategori (termasuk yang kosong):
formatted_experience_level
NaN                 2947
Mid-Senior level     539
Associate            315
Entry level          178
Director              51
Internship            33
Executive             22
Name: count, dtype: int64


Terdapat 2.947 data kosong (NaN). Ini artinya sekitar 70% lowongan tidak mencantumkan tingkat pengalaman secara formal. data kosong nantinya diisi dengan tidak terdefinisi

Cek inkonsistensi description

In [10]:
# 1. Hitung statistik dasar
print("--- Statistik Panjang Teks ---")
print(f"Rata-rata karakter Description: {df['description'].str.len().mean():.2f}")
print(f"Rata-rata karakter Skills Desc : {df['skills_desc'].str.len().mean():.2f}")

# 2. Filter data yang deskripsinya di bawah 50 karakter
# Kita ambil kolom title dan location juga supaya tahu ini lowongan apa
pendek_df = df[df['description'].str.len() < 50][['title', 'location', 'description']]

print(f"\n--- Daftar Data Deskripsi Terlalu Pendek (< 50 Karakter) ---")
print(f"Total ditemukan: {len(pendek_df)} data")

# 3. Tampilkan semua data yang pendek tersebut
if not pendek_df.empty:
    display(pendek_df)
else:
    print("Tidak ditemukan deskripsi yang terlalu pendek.")

--- Statistik Panjang Teks ---
Rata-rata karakter Description: 2178.39
Rata-rata karakter Skills Desc : 36.72

--- Daftar Data Deskripsi Terlalu Pendek (< 50 Karakter) ---
Total ditemukan: 4 data


,title,location,description
810,??????,"Jakarta, Jakarta, Indonesia",/ 5 HR + 3 / 0 1 HR / 0 1 CEO HR + HR
1069,Finance Specialist,"South Jakarta, Jakarta, Indonesia",财务专员：财务专业，二年以上工作经验，中文口语和读写能力较好
2551,Housekeeping & enggineering,Greater Denpasar,Minimum 1 Years Experience
2702,Staff Cost & Productivity,"Kebon Jeruk, Jakarta, Indonesia",monitoring productivity


Terahkir, cek inkonsistensi kata pada desc_skill

In [11]:
# 1. Cek Statistik Panjang Karakter untuk membantu mendeteksi apakah isinya cuma teks pendek/sampah)
skills_len = df['skills_desc'].str.len()
print(f"Rata-rata panjang karakter: {skills_len.mean():.2f}")
print(f"Panjang karakter terpendek: {skills_len.min()}")
print(f"Panjang karakter terpanjang: {skills_len.max()}")

# 2. Tampilkan data yang isinya sangat pendek (misal < 20 karakter)
print("\n--- Contoh Data skills_desc yang Terlalu Pendek (< 20 Karakter) ---")
pendek_skills = df[df['skills_desc'].str.len() < 20][['title', 'skills_desc']]
if not pendek_skills.empty:
    display(pendek_skills.head(10))
else:
    print("Tidak ada data yang terlalu pendek.")

Rata-rata panjang karakter: 36.72
Panjang karakter terpendek: 2.0
Panjang karakter terpanjang: 184.0

--- Contoh Data skills_desc yang Terlalu Pendek (< 20 Karakter) ---


,title,skills_desc
370,Biomedical Imaging & Segmentation Specialist |...,3D Slicer
406,Mechanical Engineer (Autodesk) | Remote,inventor
407,Mechanical Engineer (Autodesk) | Remote,inventor
521,Fire Service Supervisor | $75/hr | Remote,firefighter
532,System Administrator | Remote,cloud
533,Cloud Engineer | Remote,cloud
534,Senior System Administrator | Remote,cloud
580,Statistical Specialist | Remote,consultant
928,Technical Architect | Remote,Architect
1206,HR Policy & Compliance Specialist | $36/hr Remote,Human Resources


aman, tidak ada data yang aneh

check **missing value** pada job_postings.csv


In [12]:
# Mengecek total missing values dan persentasenya
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

# Menggabungkan hasil dalam satu DataFrame
missing_df = pd.DataFrame({
    'Total Missing': missing_values,
    'Percentage Missing (%)': missing_percentage
})

# Menampilkan hanya kolom yang memiliki missing values, diurutkan dari yang terbanyak
missing_df = missing_df[missing_df['Total Missing'] > 0].sort_values(by="Total Missing", ascending=False)

# Menampilkan hasil
missing_df

,Total Missing,Percentage Missing (%)
skills_desc,4011,98.188494
formatted_experience_level,2947,72.141983
application_url,2252,55.128519
company_id,57,1.395349
description,10,0.244798


Selanjutnya adalah check duplicated data pada main_data.csv

In [13]:
total_duplicates = df.duplicated().sum()

print(f"Total duplicated rows: {total_duplicates}")

if total_duplicates > 0:
    duplicated_data = df[main_data_df.duplicated()]
    display(duplicated_data)

Total duplicated rows: 0


Selanjutnya cek anommali teks

In [15]:
# [1] Cek Anomali Panjang Deskripsi
min_len = 100
max_len = 5000

print(f"--- [1] Anomali Panjang Teks ---")
anomali_pendek = df[df['description'].str.len() < min_len]
anomali_panjang = df[df['description'].str.len() > max_len]

print(f"Deskripsi < {min_len} karakter: {len(anomali_pendek)} data")
print(f"Deskripsi > {max_len} karakter: {len(anomali_panjang)} data")

# Tampilkan yang terlalu pendek (biasanya ini yang paling bermasalah)
if not anomali_pendek.empty:
    display(anomali_pendek[['title', 'description']].head(10))

--- [1] Anomali Panjang Teks ---
Deskripsi < 100 karakter: 8 data
Deskripsi > 5000 karakter: 256 data


,title,description
399,Gerente assistente/líder de equipe – com detal...,Organization- Park Hyatt Jakarta\n\nResumo\n\n...
810,??????,/ 5 HR + 3 / 0 1 HR / 0 1 CEO HR + HR
1069,Finance Specialist,财务专员：财务专业，二年以上工作经验，中文口语和读写能力较好
2452,Human Resources Executive,Bachelor Degree in law majoring\nAt least 5 ye...
2551,Housekeeping & enggineering,Minimum 1 Years Experience
2702,Staff Cost & Productivity,monitoring productivity
2828,Procurement Service ar Smelter,"Sarjana Teknik, dengan pengalaman min 7 tahun ..."
2831,Financial Advisor,- Luwes dan dapat berkomunikasi dengan baik.\n...


In [16]:
# [2] Cek Anomali Tag HTML (Noise)
import re

def has_html(text):
    pattern = re.compile('<.*?>') # Mencari pola <...tag...>
    return bool(pattern.search(str(text)))

print(f"--- [2] Anomali Noise (HTML) ---")
anomali_html = df[df['description'].apply(has_html)]

print(f"Data dengan sisa tag HTML: {len(anomali_html)} data")
if not anomali_html.empty:
    display(anomali_html[['title', 'description']].head(5))

--- [2] Anomali Noise (HTML) ---
Data dengan sisa tag HTML: 2 data


,title,description
2556,Staff cost control,"Hallo Everyone, Sushi Stop adalah restaurant J..."
3623,Business Executive Allianz,🌍 We’re Hiring – Business Executive 🌍Tired of ...


In [17]:
!pip install langdetect
from langdetect import detect
import time

def check_lang_combined(row):
    cols_to_check = [
        'title', 'description', 'skills_desc', 'location',
        'formatted_work_type', 'formatted_experience_level'
    ]

    # Gabungkan semua isi kolom tersebut menjadi satu string panjang
    combined_text = ""
    for col in cols_to_check:
        if col in row and not pd.isna(row[col]):
            combined_text += str(row[col]) + " "

    try:
        # Jika string kosong setelah digabung, return unknown
        if not combined_text.strip():
            return "unknown"
        # Deteksi bahasa dari gabungan teks tersebut
        return detect(combined_text)
    except:
        return "unknown"

print("--- [3] Memulai Deteksi Bahasa Global (Seluruh Teks & Semua Kolom) ---")
print("Proses ini akan mengecek Judul, Lokasi, Deskripsi, hingga Level secara bersamaan...")

start_time = time.time()

# Menjalankan deteksi pada gabungan kolom per baris (axis=1)
df['detected_lang_global'] = df.apply(check_lang_combined, axis=1)

end_time = time.time()
durasi = end_time - start_time

print(f"\nSelesai dalam {durasi:.2f} detik.")

# 1. Tampilkan Statistik Bahasa
print("\nStatistik Bahasa yang Terdeteksi secara Global:")
print(df['detected_lang_global'].value_counts())

# 2. Filter Anomali (Selain Indonesia 'id' dan Inggris 'en')
anomali_global = df[~df['detected_lang_global'].isin(['id', 'en'])]

print(f"\n--- HASIL ANOMALI BAHASA GLOBAL ---")
print(f"Jumlah data anomali ditemukan: {len(anomali_global)}")

if not anomali_global.empty:
    # Tampilkan kolom-kolom utama untuk pengecekan
    display(anomali_global[['title', 'location', 'detected_lang_global', 'description']])
else:
    print("Sempurna! Seluruh kolom di semua baris hanya berisi Bahasa Indonesia atau Inggris.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=9e04f74699a8f154abf1d9eedbe5b1cee7ead9788d67d70d694a25daf9c2383d
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
--- [3] Memulai Deteksi Bahasa Global (Seluruh Teks & Semua Kolom) ---
Proses ini akan mengecek Judul, Lokasi, Deskripsi, hingga Level secara bersamaan...

Selesai dalam 34.23 detik.

Statistik Bahasa yang Terdeteksi secara Global:
detected_lang_global
en         3285
id          738
zh-cn        30
nl           10
pt            4
ja            3
ko            3
it            2
de            2
fr            2
fi            1
da            1
vi            1
ca            1
unknown       1
bg            1
Name: count, dtype: int64

--- HASIL ANOMALI BAHASA GLOBAL ---
Jumlah dat

,title,location,detected_lang_global,description
32,SAPテクノロジー_AI & Technology Modernization Busine...,"Lembang, West Java, Indonesia",ja,組織について\n\n30年近くにわたるSAP社との強固なパートナーシップと最新テクノロジーへ...
143,反欺诈风控运营高级专员/专家,APAC,zh-cn,注：该岗位需要relocate到阿布扎比或马来西亚工作\n关于我们Bybit 成立于 201...
180,Assistent Teamleader Kassa en Service,"Balen, East Java, Indonesia",nl,Assistent teamleader kassa en service in Balen...
182,Managed Services Consultant Junior,"Laren, East Java, Indonesia",nl,Ben jij een nieuwsgierige data‑professional di...
399,Gerente assistente/líder de equipe – com detal...,"Jakarta, Jakarta, Indonesia",pt,Organization- Park Hyatt Jakarta\n\nResumo\n\n...
...,...,...,...,...
3592,销售经理,"Lembang, West Java, Indonesia",zh-cn,全国\n\n首页职位校园海归 简历优化 猎聘APP 我是猎头我是招聘方 NEW 登录/注册\...
3593,软件项目工程师,"Lembang, West Java, Indonesia",zh-cn,全国\n\n首页职位校园海归 简历优化 猎聘APP 我是猎头我是招聘方 NEW 登录/注册\...
3926,Стажант в Правен отдел,"Lembang, West Java, Indonesia",bg,Готов/а ли си да направиш първите стъпки към с...
3954,Weekendhulp - Laren,"Laren, East Java, Indonesia",nl,Anna van Toor gelooft in (h)echte vriendschapp...


**Insight**

Setelah melakukan proses Assessing Data, berikut beberapa temuan dan wawasan yang dapat diambil sebelum masuk ke tahap Cleaning Data:
1. Inkonsistensi data
- Terdapat 7 data pada pada kolom judul yang mengandung gaji (Rp,$)
- Pada kolom location terdapat 3 lokasi yaitu Indonesia, APAC, dan APJ terdeteksi sebagai data yang terlalu general. Sesuai rencana, data ini akan dikategorikan sebagai "Remote/Regional" untuk menjaga relevansi filter lokasi bagi pengguna.
- Pada kolom deskripsi ada 1 baris yang tidak ada judul/title nya
2. Missing Value
- Pada kolom skill_desc, missing value sangat besar 98%, formatted_experience_level 72%, application_url 55%, company_id 1% dan description 0,2
-Strategi: Kolom skill_desc akan diabaikan karena informasi serupa dapat diekstraksi dari kolom description. Untuk experience_level, akan dilakukan Feature Engineering menggunakan keyword matching pada deskripsi pekerjaan untuk memulihkan data yang hilang. Kolom description memiliki missing value sangat rendah (0,2%). Namun, karena ini adalah data inti, baris yang kosong akan langsung dihapus.
Data Vital: Kolom description memiliki missing value sangat rendah (0,2%). Namun, karena ini adalah data inti, baris yang kosong akan langsung dihapus
- Langkah selanjutnya adalah menentukan strategi menghapus tau mengisi (imputasi) missing values berdasarkan pola yang ditemukan
3. Anomali teks
- Terdapat 2 data yang memiliki deskripsi tidak sesuai/jelas (index 810 &index 1069)
- Terdapat 2 data yang  memiliki sisa tag html
- Terdapat 63 data yang menggunakan bahasa selain Indonesia dan Inggris, data tersebut akan dihapus karena tidak relevan dengan pekerjaan yang ada di Indonesia, yang hanya akan menjadi noise (sampah)

#**Data Cleaning**

1. Menangani Inkonsistensi Data

In [43]:
salary_pattern = r'\s?[|@-]*\s?(@?\s?(\$|Rp|IDR|USD)\s?[\d\.,]+.*|@\s?[\d\.,]+.*)'
df['title'] = df['title'].str.replace(salary_pattern, '', regex=True).str.strip()
df['title'] = df['title'].str.replace(r'[|/-]$', '', regex=True).str.strip()
# cek apa masih ada judul yang memiliki karakter @,$, RP,IDR
currency_pattern = r'(\$\s?\d+|Rp\s?\d+|IDR\s?\d+)'
title_with_salary = df[df['title'].str.contains(currency_pattern, regex=True, na=False)]
print(f"Total ditemukan: {len(title_with_salary)} data\n")

if not title_with_salary.empty:
    # Menampilkan judul agar kita bisa lihat polanya
    display(title_with_salary[['title', 'location']])
else:
    print("Data Bersih")

Total ditemukan: 0 data

Data Bersih


/tmp/ipykernel_11510/1621548691.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  title_with_salary = df[df['title'].str.contains(currency_pattern, regex=True, na=False)]


In [31]:
# Mengubah Lokasi Indonesia, APAC, dan APJ menjadi remote. Karena, lokasi kurang spesifik/jelas.
remote_keywords = ['Indonesia', 'APAC', 'APJ', 'Southeast Asia']
df['location'] = df['location'].apply(lambda x: 'Remote/Nasional' if x in remote_keywords else x)

2. Menangani Missing Value
Mengecek kembali Missing value di dataset dan menentukan strategi penanganannya

In [18]:
# Mengecek total missing values dan persentasenya
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

# Menggabungkan hasil dalam satu DataFrame
missing_df = pd.DataFrame({
    'Total Missing': missing_values,
    'Percentage Missing (%)': missing_percentage
})

# Menampilkan hanya kolom yang memiliki missing values, diurutkan dari yang terbanyak
missing_df = missing_df[missing_df['Total Missing'] > 0].sort_values(by="Total Missing", ascending=False)

# Menampilkan hasil
missing_df

,Total Missing,Percentage Missing (%)
skills_desc,4011,98.188494
formatted_experience_level,2947,72.141983
application_url,2252,55.128519
company_id,57,1.395349
description,10,0.244798


Missing Values Sangat Tinggi (>50%) -> Hapus

Jika suatu kolom memiliki lebih dari 50% data yang hilang, maka kemungkinan besar informasi di dalamnya tidak cukup berguna untuk analisis.


Syarat:

Kolom memiliki missing values lebih dari 50% dari total data.
Kolom tidak bisa diimputasi dengan cara yang masuk akal.
Kolom tidak memiliki dampak signifikan terhadap analisis, karena data skill sudah tersedia di kolom description


Tindakan:

Hapus kolom tersebut dari dataset.
Kolom yang Dihapus:

skill_desc (98.18% missing)

In [25]:
df.drop(columns=['skills_desc'], inplace=True,errors='ignore')
print(df.columns.tolist())

['job_id', 'company_id', 'title', 'description', 'formatted_work_type', 'location', 'job_posting_url', 'application_url', 'formatted_experience_level', 'work_type', 'detected_lang_global']


Melakukan imputasi pada kolom application_url "No URL Provided"

In [26]:
df['application_url'] = df['application_url'].fillna('No URL Provided')

Missing Values Sedang (1% - 50%) -> Imputasi atau Hapus

Jika suatu kolom memiliki missing values antara 1% - 50%, maka kita bisa melakukan imputasi atau menghapusnya tergantung pada kasusnya.

Syarat:

Kolom memiliki missing values antara 1% - 50% dari total data.
Jika data kategori, bisa diisi dengan mode (kategori yang paling sering muncul).
Jika data numerik, bisa diisi dengan median atau mean, tergantung distribusinya.

Tindakan:

Imputasi menggunakan median, mean, atau mode.
Jika tidak bisa diimputasi dengan masuk akal, hapus baris yang memiliki missing values.

Kolom yang diimputasi :
- company_id

In [27]:
# Mengisi company_id yang kosong dengan 0
df['company_id'] = df['company_id'].fillna(0)

Missing Values Rendah (<1%) -> Imputasi atau Hapus Baris
Jika suatu kolom hanya memiliki missing values kurang dari 1%, kita bisa langsung mengisi data tersebut atau menghapus barisnya jika jumlahnya sangat kecil.

Syarat:

Kolom memiliki missing values kurang dari 1% dari total data.
Data masih bisa diimputasi tanpa merusak kualitas dataset.
Jika jumlah missing values sangat kecil (misalnya <0.1%), bisa dihapus saja.

Tindakan:

Imputasi numerik → median atau mean.

Imputasi kategori → mode.
Jika jumlah missing sangat sedikit, bisa hapus barisnya.

Kolom yang Diimputasi atau Dihapus:
description

In [28]:
df.dropna(subset=['description'], inplace=True)

In [29]:
# Mengecek total missing values dan persentasenya
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

# Menggabungkan hasil dalam satu DataFrame
missing_df = pd.DataFrame({
    'Total Missing': missing_values,
    'Percentage Missing (%)': missing_percentage
})

# Menampilkan hanya kolom yang memiliki missing values, diurutkan dari yang terbanyak
missing_df = missing_df[missing_df['Total Missing'] > 0].sort_values(by="Total Missing", ascending=False)

# Menampilkan hasil
missing_df

,Total Missing,Percentage Missing (%)
formatted_experience_level,2937,72.07362


**3. Anomali Teks**

1. Menghapus data yang tidak menggunakan Bahasa Indonesia atau Inggris

In [44]:
df = df[df['detected_lang_global'].isin(['id', 'en'])]

In [45]:
from langdetect import detect
import time

def check_lang_combined(row):
    cols_to_check = [
        'title', 'description', 'skills_desc', 'location',
        'formatted_work_type', 'formatted_experience_level'
    ]

    # Gabungkan semua isi kolom tersebut menjadi satu string panjang
    combined_text = ""
    for col in cols_to_check:
        if col in row and not pd.isna(row[col]):
            combined_text += str(row[col]) + " "

    try:
        # Jika string kosong setelah digabung, return unknown
        if not combined_text.strip():
            return "unknown"
        # Deteksi bahasa dari gabungan teks tersebut
        return detect(combined_text)
    except:
        return "unknown"

print("--- [3] Memulai Deteksi Bahasa Global (Seluruh Teks & Semua Kolom) ---")
print("Proses ini akan mengecek Judul, Lokasi, Deskripsi, hingga Level secara bersamaan...")

start_time = time.time()

# Menjalankan deteksi pada gabungan kolom per baris (axis=1)
df['detected_lang_global'] = df.apply(check_lang_combined, axis=1)

end_time = time.time()
durasi = end_time - start_time

print(f"\nSelesai dalam {durasi:.2f} detik.")

# 1. Tampilkan Statistik Bahasa
print("\nStatistik Bahasa yang Terdeteksi secara Global:")
print(df['detected_lang_global'].value_counts())

# 2. Filter Anomali (Selain Indonesia 'id' dan Inggris 'en')
anomali_global = df[~df['detected_lang_global'].isin(['id', 'en'])]

print(f"\n--- HASIL ANOMALI BAHASA GLOBAL ---")
print(f"Jumlah data anomali ditemukan: {len(anomali_global)}")

if not anomali_global.empty:
    # Tampilkan kolom-kolom utama untuk pengecekan
    display(anomali_global[['title', 'location', 'detected_lang_global', 'description']])
else:
    print("Sempurna! Seluruh kolom di semua baris hanya berisi Bahasa Indonesia atau Inggris.")

--- [3] Memulai Deteksi Bahasa Global (Seluruh Teks & Semua Kolom) ---
Proses ini akan mengecek Judul, Lokasi, Deskripsi, hingga Level secara bersamaan...

Selesai dalam 35.81 detik.

Statistik Bahasa yang Terdeteksi secara Global:
detected_lang_global
en         3279
id          734
unknown       1
Name: count, dtype: int64

--- HASIL ANOMALI BAHASA GLOBAL ---
Jumlah data anomali ditemukan: 1


/tmp/ipykernel_11510/4043251445.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['detected_lang_global'] = df.apply(check_lang_combined, axis=1)


,title,location,detected_lang_global,description
104,Solutions Architect for Automotive,Remote/Nasional,unknown,Canonical is a leading provider of open source...


setelah dilakukan pengecakan, terdapat 1 data yang menggunakan bahasa global, namun jika dicek manual, itu adalah gabungan bahsa Indonesia dan Inggris didalam satu baris

---



---



In [54]:
import re

def clean_html(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '', str(text))
df.loc[:, 'description'] = df['description'].apply(clean_html)

print("Tag HTML berhasil dihapus.")

Tag HTML berhasil dihapus secara permanen.


cek apakah data dengan tag html sudah hilang

In [55]:
import re
def has_html(text):
    pattern = re.compile('<.*?>') # Mencari pola <...tag...>
    return bool(pattern.search(str(text)))

anomali_html = df[df['description'].apply(has_html)]
print(f"Data dengan sisa tag HTML: {len(anomali_html)} data")
if not anomali_html.empty:
    display(anomali_html[['title', 'description']].head(5))

Data dengan sisa tag HTML: 0 data


## Feature Engineering

Kolom formatted_experience_level memiliki kekosongan yang sangat tinggi (72%). Menghapus kolom ini akan menghilangkan data penting (misal Senior vs Junior).

Tindakan: Heuristic Labeling melalui Keyword Matching. Mengekstraksi informasi dari gabungan kolom title dan description untuk mengisi nilai yang kosong.

Alasan: Informasi tingkat pengalaman sering kali tersirat dalam teks (misal: kata "Senior", "Intern", atau "Manager"). Dengan teknik ini, kelengkapan data meningkat dari 28% menjadi 100%, yang secara signifikan memperkuat basis data untuk model pencocokan (matching).



In [58]:
def fill_experience_level(row):
    # Jika sudah ada isinya (bukan NaN), gunakan isi yang sudah ada
    if pd.notna(row['formatted_experience_level']):
        return row['formatted_experience_level']

    # Gabungkan judul dan deskripsi untuk dicari kata kuncinya
    text = f"{str(row['title'])} {str(row['description'])}".lower()
    if any(k in text for k in ['intern', 'internship', 'magang', 'co-op']):
        return 'Internship'
    elif any(k in text for k in ['director', 'vp', 'vice president', 'head of', 'chief', 'executive']):
        return 'Executive'
    elif any(k in text for k in ['manager', 'lead', 'principal', 'spv', 'supervisor']):
        return 'Mid-Senior level'
    elif any(k in text for k in ['senior', 'sr.', 'expert', 'specialist']):
        return 'Mid-Senior level'
    elif any(k in text for k in ['associate', 'staff', 'junior', 'jr.', 'entry level', 'fresh graduate', 'graduates']):
        return 'Entry level'

    # Jika benar-benar tidak ditemukan, beri label 'Not Applicable'
    return 'Not Applicable'
df.loc[:, 'formatted_experience_level'] = df.apply(fill_experience_level, axis=1)
print(df['formatted_experience_level'].value_counts())

formatted_experience_level
Mid-Senior level    1277
Internship          1197
Not Applicable       539
Entry level          377
Associate            315
Executive            259
Director              50
Name: count, dtype: int64


#Text Preprocessing
Teks mentah mengandung banyak "noise" yang dapat menurunkan akurasi algoritma NLP (IndoBERT).

Tindakan:
1.  Case Folding: Mengubah teks menjadi huruf kecil (lowercase).

2.  Cleaning: Menghapus angka, tanda baca, simbol gaji ($, Rp), dan tag HTML.
3.  Tokenizing & Stopwords Removal: Memecah kalimat menjadi kata dan menghapus kata umum (seperti: "yang", "dan", "the", "is") dari Bahasa Indonesia dan Inggris.

Alasan: Mengurangi dimensi data dan memastikan model hanya fokus pada kata-kata bermakna (seperti nama skill atau jabatan). Hal ini meningkatkan Signal-to-Noise Ratio, sehingga proses perhitungan kemiripan (similarity score) menjadi jauh lebih akurat.

In [62]:
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

df = df.copy()
stop_words_id = set(stopwords.words('indonesian'))
stop_words_en = set(stopwords.words('english'))
all_stopwords = stop_words_id.union(stop_words_en)

def preprocess_text(text):
    if pd.isna(text) or text == "":
        return ""
    # a. Mengubah ke Lowercase
    text = str(text).lower()
    # b. Menghapus Angka
    text = re.sub(r'\d+', '', text)
    # c. Menghapus Tanda Baca
    text = text.translate(str.maketrans('', '', string.punctuation))
    # d. Tokenisasi
    tokens = word_tokenize(text)
    # e. Hapus Stopwords & Kata Pendek (< 3 huruf)
    cleaned_tokens = [w for w in tokens if w not in all_stopwords and len(w) > 2]
    return " ".join(cleaned_tokens)
df['cleaned_description'] = df['description'].apply(preprocess_text)
df['cleaned_title'] = df['title'].apply(preprocess_text)

print("--- TEXT PREPROCESSING SELESAI ---")
display(df[['title', 'cleaned_title', 'cleaned_description']].head())

--- TEXT PREPROCESSING SELESAI ---


,title,cleaned_title,cleaned_description
0,Sales Executive,sales executive,smart authentic proactive sales professionalar...
1,Partnership Associate,partnership associate,responsibilities building strong strategic rel...
2,Human Resources Coordinator,human resources coordinator,hiring fleet supervisor surya mitra tirta kenc...
3,Finance Accounting Specialist,finance accounting specialist,job description engage monthly financial close...
4,Sales Marketing for Marine & Offshore,sales marketing marine offshore,kualifikasipendidikan minimal bidang pemasaran...


Membuat kolom baru cleaned_features yang merupakan gabungan dari cleaned_title dan cleaned_description.

In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4014 entries, 0 to 4084
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   job_id                      4014 non-null   int64  
 1   company_id                  4014 non-null   float64
 2   title                       4014 non-null   object 
 3   description                 4014 non-null   object 
 4   formatted_work_type         4014 non-null   object 
 5   location                    4014 non-null   object 
 6   job_posting_url             4014 non-null   object 
 7   application_url             4014 non-null   object 
 8   formatted_experience_level  4014 non-null   object 
 9   work_type                   4014 non-null   object 
 10  detected_lang_global        4014 non-null   object 
 11  cleaned_description         4014 non-null   object 
 12  cleaned_title               4014 non-null   object 
dtypes: float64(1), int64(1), object(11)
me

In [64]:
from google.colab import files

# Pastikan index urut dari 0
df.reset_index(drop=True, inplace=True)

# Simpan ke CSV
df.to_csv('job_postings_final_cleaned.csv', index=False)

# Download ke perangkat kamu
files.download('job_postings_final_cleaned.csv')

print("File CSV sedang diunduh...")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File CSV sedang diunduh...
